# ERCOT Weather Data Download from Open-Meteo

This notebook downloads hourly weather data (temperature, wind speed, solar radiation) 
for each ERCOT zone using one representative city per zone.

## Zone → City Mapping
| Zone | City | Rationale |
|------|------|----------|
| COAST | Houston | Major coastal load center |
| EAST | Lufkin | Center of East Texas |
| FWEST | Abilene | Far West TX, wind/oil region |
| NORTH | Wichita Falls | Northern boundary zone |
| NCENT | Dallas | Largest load zone in ERCOT |
| SOUTH | San Antonio | Major southern city |
| SCENT | Austin | South-central hub |
| WEST | Midland | West Texas desert region |

## Step 1: Install Required Libraries

We use `openmeteo-requests` (the official Open-Meteo Python client) and `requests-cache` 
to avoid re-downloading data if you run the notebook again.

## Step 2: Import Libraries & Set Up API Client

In [6]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd
from datetime import datetime, timedelta, timezone

# Set up a cached session so repeated runs don't re-hit the API
# expire_after=-1 means cache never expires (good for historical data that won't change)
cache_session = requests_cache.CachedSession('.openmeteo_cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

print("API client ready!")

API client ready!


## Step 3: Define ERCOT Zones with Representative Coordinates

In [7]:
# Each ERCOT zone mapped to one representative city
# Format: zone_name -> (city_name, latitude, longitude)
ERCOT_ZONES = {
    "COAST":  ("Houston",        29.7604, -95.3698),
    "EAST":   ("Lufkin",         31.3382, -94.7291),
    "FWEST":  ("Abilene",        32.4487, -99.7331),
    "NORTH":  ("Wichita Falls",  33.9137, -98.4934),
    "NCENT":  ("Dallas",         32.7767, -96.7970),
    "SOUTH":  ("San Antonio",    29.4241, -98.4936),
    "SCENT":  ("Austin",         30.2672, -97.7431),
    "WEST":   ("Midland",        31.9973, -102.0779),
}

print("ERCOT zones defined:")
for zone, (city, lat, lon) in ERCOT_ZONES.items():
    print(f"  {zone:6s} → {city} ({lat}, {lon})")

ERCOT zones defined:
  COAST  → Houston (29.7604, -95.3698)
  EAST   → Lufkin (31.3382, -94.7291)
  FWEST  → Abilene (32.4487, -99.7331)
  NORTH  → Wichita Falls (33.9137, -98.4934)
  NCENT  → Dallas (32.7767, -96.797)
  SOUTH  → San Antonio (29.4241, -98.4936)
  SCENT  → Austin (30.2672, -97.7431)
  WEST   → Midland (31.9973, -102.0779)


## Step 4: Set Date Range

We use **2018–2025** to match our train/validate/test split:
- Train: 2018–2023
- Validate: 2024
- Test: 2025

In [8]:
START_DATE = "2018-01-01"
END_DATE   = "2025-12-31"

print(f"Downloading data from {START_DATE} to {END_DATE}")
print(f"That's approximately {(pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days * 24:,} hourly rows per zone")

That's approximately 70,104 hourly rows per zone


## Step 5: Download Function

This function hits the Open-Meteo **Historical Weather API** for a single zone 
and returns a clean DataFrame.

**Variables we're downloading:**
- `temperature_2m` — air temperature at 2 meters above ground (°C)
- `relative_humidity_2m` — humidity (%)
- `wind_speed_10m` — wind speed at 10m height (km/h)
- `wind_direction_10m` — wind direction in degrees
- `shortwave_radiation` — solar radiation (W/m²) — proxy for solar generation potential
- `direct_normal_irradiance` — direct solar beam radiation (W/m²)

In [6]:
def download_zone_weather(zone_name, city_name, lat, lon, start_date, end_date):
    """
    Download hourly weather data for one ERCOT zone from Open-Meteo.
    Returns a pandas DataFrame with a datetime index in US/Central time.
    """
    print(f"Downloading {zone_name} ({city_name})...", end=" ")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "wind_direction_10m",
            "shortwave_radiation",        # total solar radiation → solar generation proxy
            "direct_normal_irradiance",   # direct beam radiation
        ],
        "timezone": "America/Chicago",   # ERCOT operates on Central Time
        "temperature_unit": "fahrenheit", # keep in °F to match ERCOT convention
        "wind_speed_unit": "mph",
    }
    
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    
    # Extract hourly data
    hourly = response.Hourly()
    
    # Build time index
    start_ts = pd.Timestamp(hourly.Time(), unit="s", tz="UTC").tz_convert("America/Chicago")
    end_ts   = pd.Timestamp(hourly.TimeEnd(), unit="s", tz="UTC").tz_convert("America/Chicago")
    freq     = f"{hourly.Interval()}s"
    time_index = pd.date_range(start=start_ts, end=end_ts, freq=freq, inclusive="left")
    
    # Build DataFrame — variable order must match the 'hourly' list in params above
    df = pd.DataFrame({
        "datetime":               time_index,
        "temperature_f":          hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_pct":  hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed_mph":         hourly.Variables(2).ValuesAsNumpy(),
        "wind_direction_deg":     hourly.Variables(3).ValuesAsNumpy(),
        "solar_radiation_wm2":    hourly.Variables(4).ValuesAsNumpy(),
        "direct_irradiance_wm2":  hourly.Variables(5).ValuesAsNumpy(),
    })
    
    df["zone"] = zone_name
    df["city"] = city_name
    df = df.set_index("datetime")
    
    print(f"Done! {len(df):,} rows downloaded.")
    return df

## Step 6: Download All 8 Zones

This will take a few minutes the first time, but results are cached so re-runs are instant.

In [7]:
import time

all_zones = []
zone_list = list(ERCOT_ZONES.items())

for i, (zone_name, (city_name, lat, lon)) in enumerate(zone_list):
    df_zone = download_zone_weather(
        zone_name=zone_name,
        city_name=city_name,
        lat=lat,
        lon=lon,
        start_date=START_DATE,
        end_date=END_DATE
    )
    all_zones.append(df_zone)
    
    # Wait between requests — skip the pause after the last zone
    if i < len(zone_list) - 1:
        wait_seconds = 70
        print(f"  Waiting {wait_seconds}s before next zone to respect rate limit...")
        for remaining in range(wait_seconds, 0, -10):
            print(f"    {remaining}s remaining...", end="\r")
            time.sleep(10)
        print("  Ready for next zone!                    ")

print("\nAll zones downloaded successfully!")

  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    
  Waiting 70s before next zone to respect rate limit...
  Ready for next zone!                    

All zones downloaded successfully!


## Step 7: Combine into One Master DataFrame

We stack all zones into one long-format DataFrame. This format is easy to merge with ERCOT load data later.

In [8]:
# Combine all zone DataFrames into one
df_all = pd.concat(all_zones)
df_all = df_all.reset_index()  # bring datetime back as a column

# Reorder columns cleanly
df_all = df_all[[
    "datetime", "zone", "city",
    "temperature_f", "relative_humidity_pct",
    "wind_speed_mph", "wind_direction_deg",
    "solar_radiation_wm2", "direct_irradiance_wm2"
]]

print(f"Combined DataFrame shape: {df_all.shape}")
print(f"\nDate range: {df_all['datetime'].min()} → {df_all['datetime'].max()}")
print(f"\nZones present: {df_all['zone'].unique().tolist()}")
print(f"\nSample data:")
df_all.head(10)

Combined DataFrame shape: (561024, 9)

Date range: 2018-01-01 00:00:00-06:00 → 2025-12-31 23:00:00-06:00

Zones present: ['COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST']

Sample data:


,datetime,zone,city,temperature_f,relative_humidity_pct,wind_speed_mph,wind_direction_deg,solar_radiation_wm2,direct_irradiance_wm2
0,2018-01-01 00:00:00-06:00,COAST,Houston,37.400002,61.423840,14.218595,24.145542,0.0,0.000000
1,2018-01-01 01:00:00-06:00,COAST,Houston,37.130001,56.940586,14.627995,23.428713,0.0,0.000000
2,2018-01-01 02:00:00-06:00,COAST,Houston,36.230000,55.740044,14.627995,23.428713,0.0,0.000000
3,2018-01-01 03:00:00-06:00,COAST,Houston,35.419998,54.566811,13.129875,23.070442,0.0,0.000000
4,2018-01-01 04:00:00-06:00,COAST,Houston,34.970001,50.658669,13.706075,26.146788,0.0,0.000000
5,2018-01-01 05:00:00-06:00,COAST,Houston,33.169998,47.690674,14.041520,30.650600,0.0,0.000000
6,2018-01-01 06:00:00-06:00,COAST,Houston,32.180000,54.279446,13.733430,12.225114,0.0,0.000000
7,2018-01-01 07:00:00-06:00,COAST,Houston,31.549999,50.901672,15.775212,18.178116,0.0,0.000000
8,2018-01-01 08:00:00-06:00,COAST,Houston,30.469999,48.958817,15.148967,16.294130,8.0,9.940964
9,2018-01-01 09:00:00-06:00,COAST,Houston,30.380001,46.300640,15.002917,20.056185,107.0,136.501724


## Step 8: Add Calendar Features

We add these directly since we already have the datetime column:
- Hour of day (0–23)
- Day of week (0=Monday … 6=Sunday)
- Month
- Is weekend flag
- US Federal holidays (using the `holidays` library)

In [9]:
%pip install holidays -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
import holidays

# US Federal Holidays for Texas
us_holidays = holidays.US(state='TX', years=range(2018, 2026))

# Extract calendar features from the datetime column
df_all["hour"]       = df_all["datetime"].dt.hour
df_all["day_of_week"] = df_all["datetime"].dt.dayofweek   # 0=Mon, 6=Sun
df_all["month"]      = df_all["datetime"].dt.month
df_all["year"]       = df_all["datetime"].dt.year
df_all["is_weekend"] = (df_all["day_of_week"] >= 5).astype(int)  # 1 if Sat/Sun
df_all["is_holiday"] = df_all["datetime"].dt.date.astype(str).isin(
    [str(d) for d in us_holidays.keys()]
).astype(int)

print("Calendar features added!")
print(f"\nHoliday count: {df_all['is_holiday'].sum()} hourly rows flagged as holidays")
df_all[["datetime", "zone", "temperature_f", "wind_speed_mph", 
         "solar_radiation_wm2", "hour", "day_of_week", "is_weekend", "is_holiday"]].head()

Calendar features added!

Holiday count: 31296 hourly rows flagged as holidays


,datetime,zone,temperature_f,wind_speed_mph,solar_radiation_wm2,hour,day_of_week,is_weekend,is_holiday
0,2018-01-01 00:00:00-06:00,COAST,37.400002,14.218595,0.0,0,0,0,1
1,2018-01-01 01:00:00-06:00,COAST,37.130001,14.627995,0.0,1,0,0,1
2,2018-01-01 02:00:00-06:00,COAST,36.230000,14.627995,0.0,2,0,0,1
3,2018-01-01 03:00:00-06:00,COAST,35.419998,13.129875,0.0,3,0,0,1
4,2018-01-01 04:00:00-06:00,COAST,34.970001,13.706075,0.0,4,0,0,1


## Step 9: Quick Data Quality Check

In [11]:
print("=== MISSING VALUES ===")
print(df_all.isnull().sum())

print("\n=== BASIC STATISTICS BY ZONE ===")
df_all.groupby("zone")[["temperature_f", "wind_speed_mph", "solar_radiation_wm2"]].describe().round(2)

=== MISSING VALUES ===
datetime                 0
zone                     0
city                     0
temperature_f            0
relative_humidity_pct    0
wind_speed_mph           0
wind_direction_deg       0
solar_radiation_wm2      0
direct_irradiance_wm2    1
hour                     0
day_of_week              0
month                    0
year                     0
is_weekend               0
is_holiday               0
dtype: int64

=== BASIC STATISTICS BY ZONE ===


temperature_f                                                    \
              count   mean    std    min    25%    50%    75%     max   
zone                                                                    
COAST       70128.0  70.21  13.71  12.47  61.61  72.86  79.79  107.06   
EAST        70128.0  68.07  15.68  -1.30  57.20  70.16  79.16  107.96   
FWEST       70128.0  67.35  18.44  -5.80  54.05  69.08  81.41  111.20   
NCENT       70128.0  67.35  17.23  -1.93  54.86  69.35  80.42  109.22   
NORTH       70128.0  66.39  19.17 -11.02  52.25  68.09  81.23  113.81   
SCENT       70128.0  69.88  15.72   4.55  59.09  72.05  81.05  106.88   
SOUTH       70128.0  71.12  15.09   8.33  61.07  73.22  81.68  107.15   
WEST        70128.0  66.33  18.19   3.47  52.70  68.00  80.06  111.20   

      wind_speed_mph         ...               solar_radiation_wm2          \
               count   mean  ...    75%    max               count    mean   
zone                         ...                                             
COAST        70128.0   7.70  ...  10.09  47.03             70128.0  191.00   
EAST         70128.0   6.69  ...   8.44  26.32             70128.0  193.71   
FWEST        70128.0  10.27  ...  13.54  33.57             70128.0  218.08   
NCENT        70128.0   8.78  ...  11.49  33.18             70128.0  199.09   
NORTH        70128.0   9.78  ...  12.76  32.37             70128.0  209.62   
SCENT        70128.0   8.05  ...  10.67  35.48             70128.0  199.00   
SOUTH        70128.0   7.54  ...   9.89  34.45             70128.0  202.37   
WEST         70128.0  10.13  ...  12.98  37.55             70128.0  228.76   

                                              
          std  min  25%   50%    75%     max  
zone                                          
COAST  268.90  0.0  0.0   8.0  353.0  1007.0  
EAST   271.81  0.0  0.0   8.0  357.0  1008.0  
FWEST  295.10  0.0  0.0  11.0  434.0  1041.0  
NCENT  277.98  0.0  0.0   9.0  370.0  1011.0  
NORTH  286.65  0.0  0.0  10.0  409.0  1025.0  
SCENT  279.68  0.0  0.0   9.0  366.0  1025.0  
SOUTH  282.36  0.0  0.0   9.0  375.0  1018.0  
WEST   306.11  0.0  0.0  13.0  455.0  1070.0  

[8 rows x 24 columns]

## Step 10: Create Wide-Format Version (matches ERCOT load data structure)

ERCOT load data typically has one row per timestamp with zone values as separate columns.
We create a wide-format version for easy merging.

In [12]:
# Pivot to wide format: one row per datetime, columns = zone_variable
# e.g., COAST_temperature_f, NCENT_wind_speed_mph, etc.

weather_vars = [
    "temperature_f", "relative_humidity_pct", 
    "wind_speed_mph", "solar_radiation_wm2"
]

wide_dfs = []
for var in weather_vars:
    pivoted = df_all.pivot(index="datetime", columns="zone", values=var)
    pivoted.columns = [f"{zone}_{var}" for zone in pivoted.columns]
    wide_dfs.append(pivoted)

# Also add calendar features (same for all zones, just take from first)
calendar_cols = ["hour", "day_of_week", "month", "year", "is_weekend", "is_holiday"]
calendar_df = df_all[df_all["zone"] == "NCENT"].set_index("datetime")[calendar_cols]

df_wide = pd.concat(wide_dfs + [calendar_df], axis=1).reset_index()

print(f"Wide format shape: {df_wide.shape}")
print(f"Columns: {df_wide.columns.tolist()[:12]} ...")
df_wide.head()

Wide format shape: (70128, 39)
Columns: ['datetime', 'COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f', 'COAST_relative_humidity_pct', 'EAST_relative_humidity_pct', 'FWEST_relative_humidity_pct'] ...


,datetime,COAST_temperature_f,EAST_temperature_f,FWEST_temperature_f,NCENT_temperature_f,NORTH_temperature_f,SCENT_temperature_f,SOUTH_temperature_f,WEST_temperature_f,COAST_relative_humidity_pct,...,NORTH_solar_radiation_wm2,SCENT_solar_radiation_wm2,SOUTH_solar_radiation_wm2,WEST_solar_radiation_wm2,hour,day_of_week,month,year,is_weekend,is_holiday
0,2018-01-01 00:00:00-06:00,37.400002,32.810001,17.600000,21.290001,17.509998,29.030001,31.280001,16.07,61.423840,...,0.0,0.0,0.0,0.0,0,0,1,2018,0,1
1,2018-01-01 01:00:00-06:00,37.130001,30.920000,17.330000,20.480000,16.969999,28.670000,31.010000,16.16,56.940586,...,0.0,0.0,0.0,0.0,1,0,1,2018,0,1
2,2018-01-01 02:00:00-06:00,36.230000,28.760000,17.420000,19.850000,16.520000,28.400000,30.830000,15.98,55.740044,...,0.0,0.0,0.0,0.0,2,0,1,2018,0,1
3,2018-01-01 03:00:00-06:00,35.419998,27.230000,16.340000,19.220001,15.980000,28.039999,30.290001,16.52,54.566811,...,0.0,0.0,0.0,0.0,3,0,1,2018,0,1
4,2018-01-01 04:00:00-06:00,34.970001,25.970001,15.440001,18.500000,15.080000,28.039999,30.110001,16.52,50.658669,...,0.0,0.0,0.0,0.0,4,0,1,2018,0,1


## Step 11: Save Both Formats to CSV

In [13]:
# Long format — one row per (datetime, zone) — good for exploration
df_all.to_csv("ercot_weather_long_2018_2025.csv", index=False)
print(f"Saved long format: ercot_weather_long_2018_2025.csv ({df_all.shape[0]:,} rows)")

# Wide format — one row per datetime, zones as columns — good for modeling
df_wide.to_csv("ercot_weather_wide_2018_2025.csv", index=False)
print(f"Saved wide format: ercot_weather_wide_2018_2025.csv ({df_wide.shape[0]:,} rows)")

print("\nDone! Your weather data is ready to merge with ERCOT load data.")

Saved long format: ercot_weather_long_2018_2025.csv (561,024 rows)
Saved wide format: ercot_weather_wide_2018_2025.csv (70,128 rows)

Done! Your weather data is ready to merge with ERCOT load data.


In [ ]:
# Strip the timezone offset from the raw string to get a plain local datetime
# e.g. '2018-11-04 01:00:00-05:00' → '2018-11-04 01:00:00'
df_all['_local'] = pd.to_datetime(df_all['datetime'].str[:19])
df_all = df_all.sort_values('_local').reset_index(drop=True)

## Step 12: Drop fall-back duplicates
In November, 1:00 AM appears twice (once as `-05:00`, once as `-06:00`).
After stripping the offset they look identical — we keep the first and drop the second.

In [ ]:
dupes = df_all[df_all.duplicated(subset='_local', keep=False)]
print(f'Duplicate rows found: {len(dupes)}')
display(dupes[['datetime', '_local']].head(4))

before = len(df_all)
df = df_all.drop_duplicates(subset='_local', keep='first').reset_index(drop=True)
print(f'\nDropped {before - len(df)} rows → {len(df):,} rows remaining')

Duplicate rows found: 16


,datetime,_local
7368,2018-11-04 01:00:00-05:00,2018-11-04 01:00:00
7369,2018-11-04 01:00:00-06:00,2018-11-04 01:00:00
16104,2019-11-03 01:00:00-05:00,2019-11-03 01:00:00
16105,2019-11-03 01:00:00-06:00,2019-11-03 01:00:00



Dropped 8 rows → 70,120 rows remaining


## Step 13: Reindex to a full hourly grid
We build a complete hourly index and reindex against it.
The 8 spring-forward hours (2:00 AM) become NaN rows.

In [ ]:
df = df.set_index('_local')
full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
df = df.reindex(full_index)

print(f'Rows after reindex : {len(df):,}')
print(f'NaN rows inserted  : {df.isnull().any(axis=1).sum()}')

print('\n2018-03-11 around 2AM (before interpolation):')
display(df.loc['2018-03-11 00:00':'2018-03-11 04:00', ['COAST_temperature_f']])

Rows after reindex : 70,128
NaN rows inserted  : 8

2018-03-11 around 2AM (before interpolation):


,COAST_temperature_f
2018-03-11 00:00:00,70.70
2018-03-11 01:00:00,70.61
2018-03-11 02:00:00,NaN
2018-03-11 03:00:00,69.35
2018-03-11 04:00:00,68.99


## Step 14: Interpolate the missing hours
Linear interpolation fills the NaN as the midpoint between the hour before and after.
e.g. 1AM = 70.61°F, 3AM = 69.35°F → 2AM estimated as 69.98°F

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols] = df[numeric_cols].interpolate(method='linear')

print('2018-03-11 around 2AM (after interpolation):')
display(df.loc['2018-03-11 00:00':'2018-03-11 04:00', ['COAST_temperature_f']])

2018-03-11 around 2AM (after interpolation):


,COAST_temperature_f
2018-03-11 00:00:00,70.70
2018-03-11 01:00:00,70.61
2018-03-11 02:00:00,69.98
2018-03-11 03:00:00,69.35
2018-03-11 04:00:00,68.99


## Step 15: Rebuild datetime column and re-derive time features

In [ ]:
new_datetime = df.index.copy()   # save DatetimeIndex before resetting
df = df.reset_index(drop=True)

df['datetime']    = new_datetime.strftime('%Y-%m-%d %H:%M:%S')
df['hour']        = new_datetime.hour
df['day_of_week'] = new_datetime.dayofweek
df['month']       = new_datetime.month
df['year']        = new_datetime.year
df['is_weekend']  = (new_datetime.dayofweek >= 5).astype(int)

display(df.head(3))

,datetime,COAST_temperature_f,EAST_temperature_f,FWEST_temperature_f,NCENT_temperature_f,NORTH_temperature_f,SCENT_temperature_f,SOUTH_temperature_f,WEST_temperature_f,COAST_relative_humidity_pct,...,NORTH_solar_radiation_wm2,SCENT_solar_radiation_wm2,SOUTH_solar_radiation_wm2,WEST_solar_radiation_wm2,hour,day_of_week,month,year,is_weekend,is_holiday
0,2018-01-01 00:00:00,37.40,32.81,17.60,21.29,17.509998,29.03,31.28,16.07,61.423840,...,0.0,0.0,0.0,0.0,0,0,1,2018,0,1.0
1,2018-01-01 01:00:00,37.13,30.92,17.33,20.48,16.970000,28.67,31.01,16.16,56.940586,...,0.0,0.0,0.0,0.0,1,0,1,2018,0,1.0
2,2018-01-01 02:00:00,36.23,28.76,17.42,19.85,16.520000,28.40,30.83,15.98,55.740044,...,0.0,0.0,0.0,0.0,2,0,1,2018,0,1.0


In [ ]:
print(f'Rows after reindex : {len(df):,}')
print(f'NaN rows inserted  : {df.isnull().any(axis=1).sum()}')

Rows after reindex : 70,128
NaN rows inserted  : 0


## Step 16: Final validation

In [ ]:
print(f'Total rows     : {len(df):,}')
print(f'Missing values : {df[numeric_cols].isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated(subset="datetime").sum()}')

expected = pd.date_range(start=df['datetime'].min(), end=df['datetime'].max(), freq='h')
print(f'Timestamp gaps : {len(expected) - len(df)}')
print('\n✅ Clean and complete!')

df.to_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/processed_data/ercot_weather_wide_clean.csv", index=False)
print('Saved!')

Total rows     : 70,128
Missing values : 0
Duplicates     : 0
Timestamp gaps : 0

✅ Clean and complete!
Saved!
